# KV Cache Explained and Implemented

KV caching speeds up autoregressive transformer inference by avoiding repeated work.

When a decoder-only model generates the next token, it only needs:
- a **new query** for the newest token
- all **past keys** and **past values** from earlier tokens

Instead of recomputing keys and values for the whole prefix at every decoding step, we store them in a cache and reuse them.

This notebook explains:
1. what KV cache stores
2. why it makes generation faster
3. why queries are **not** cached
4. how to implement it in PyTorch
5. how to verify that cached decoding matches full decoding

## 1. What Is KV Cache?

For a decoder self-attention layer, each token representation is projected into:
- query $Q$
- key $K$
- value $V$

For a sequence of length $t$, causal self-attention computes

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V
$$

where $M$ is the causal mask.

During generation, token $t$ depends on tokens $1, 2, \dots, t$. Without caching, at step $t$ the model recomputes projections for the full prefix again.

That means we repeatedly recompute keys and values for older tokens even though they never change.

### What exactly is cached?

For **each transformer layer** and **each attention head**, we cache:
- all past keys
- all past values

If the current step produces new key/value tensors $k_t$ and $v_t$, then the cache update is:

$$
K_{\text{cache}} \leftarrow [K_{\text{cache}}; k_t], \qquad
V_{\text{cache}} \leftarrow [V_{\text{cache}}; v_t]
$$

The newest token only needs its own query $q_t$, then it attends over the concatenation of old and new keys/values.

### Why do we cache K and V, but not Q?

Queries are only needed for the token currently being processed.

Once a past token has already produced its output, we do **not** need its query again during autoregressive decoding. Future tokens need to compare their new query against all previous keys, and then use the corresponding values.

So the reusable part is:
- keys: used to compute attention scores against future queries
- values: used to build the weighted sum after attention

### Why is KV cache faster?

Suppose the prompt length is $P$ and we generate $T$ new tokens.

Without cache, the model repeatedly processes sequences of lengths:

$$
P, P+1, P+2, \dots, P+T-1
$$

So total token positions processed is:

$$
\sum_{i=0}^{T-1}(P+i)
$$

With cache, the prompt is processed once, then each new step processes only one fresh token:

$$
P + T - 1
$$

This is why cached decoding becomes much more valuable as generation gets longer.

### Important practical point

KV caching is mainly an **inference-time** optimization. During training, we usually process full sequences in parallel, so caching is typically unnecessary.

## 1.5 Complexity: What KV Cache Actually Saves

The main complexity question is: **how much work do we repeat at each decoding step?**

Assume we generate $T$ new tokens.

### Without KV cache

At decoding step $t$, the current prefix length is $t$.
If we do not cache past keys and values, we rebuild the prefix state again for that step.

That means the repeated projection / prefix-rebuild part scales roughly like:

$$
1 + 2 + 3 + \cdots + T = \frac{T(T+1)}{2} = O(T^2)
$$

So even before discussing the attention matmul itself, we are already redoing a quadratic amount of work across the whole decode.

### With KV cache

Now suppose each layer stores its past keys and values.
At step $t$, we do **not** recompute tokens $1, \dots, t-1$.
We only compute the new key/value pair for token $t$ and append it to the cache.

So the repeated K/V construction part becomes:

$$
1 + 1 + 1 + \cdots + 1 = O(T)
$$

This is the core saving from KV cache: each generated token is projected once instead of being reprojected over and over.

### Why the full decode is not completely $O(T)$

Even with caching, the new query still has to attend over all previously cached keys and values.
At step $t$, the newest query interacts with about $t$ positions:

$$
q_t K_{1:t}^T
$$

So the attention work for the new token still grows roughly like $O(t)$ per step, and over all decode steps:

$$
\sum_{t=1}^{T} t = O(T^2)
$$

So KV cache does **not** remove the fact that the context keeps growing. What it removes is the repeated recomputation of old prefix projections.

### A useful decomposition

If hidden size is $d$, then at step $t$ we can think of the cost roughly as:

Without cache:
- rebuild K/V for all $t$ tokens: about $O(td^2)$
- run attention-related computation for the current forward pass as well

With cache:
- build K/V only for the new token: about $O(d^2)$
- compare the new query against $t$ cached keys: about $O(td)$
- take the weighted sum over $t$ cached values: about $O(td)$

So with cache, the dominant per-step cost is closer to:

$$
O(d^2 + td)
$$

instead of repeatedly rebuilding the whole prefix.

### Important nuance

When people say KV cache reduces decoding from quadratic repeated work to linear repeated work, they are usually talking specifically about the **prefix recomputation / K-V rebuilding part**.

If you measure the full attention over a growing context, the decode still gets more expensive as the sequence grows. KV cache is valuable because it removes the avoidable part of that growth.

### Tiny example

Suppose we generate 4 tokens.

Without cache, the repeated prefix work looks like:

$$
1 + 2 + 3 + 4 = 10
$$

With cache, the K/V build work looks like:

$$
1 + 1 + 1 + 1 = 4
$$

That difference is exactly why KV caching reduces latency so much in practice.

In [1]:
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F


torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
class CausalSelfAttentionWithKVCache(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        x = x.view(batch_size, seq_len, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, num_heads, seq_len, head_dim = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(batch_size, seq_len, num_heads * head_dim)

    def _build_causal_mask(
        self,
        q_len: int,
        kv_len: int,
        past_len: int,
        device: torch.device,
    ) -> torch.Tensor:
        query_positions = torch.arange(past_len, past_len + q_len, device=device).unsqueeze(-1)
        key_positions = torch.arange(kv_len, device=device).unsqueeze(0)
        return key_positions <= query_positions

    def forward(
        self,
        x: torch.Tensor,
        past_key_value: tuple[torch.Tensor, torch.Tensor] | None = None,
        use_cache: bool = False,
    ) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor] | None]:
        batch_size, current_len, _ = x.shape

        q = self._split_heads(self.q_proj(x))
        k = self._split_heads(self.k_proj(x))
        v = self._split_heads(self.v_proj(x))

        past_len = 0
        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(2)
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)

        kv_len = k.size(2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        causal_mask = self._build_causal_mask(current_len, kv_len, past_len, x.device)
        scores = scores.masked_fill(~causal_mask.view(1, 1, current_len, kv_len), float("-inf"))

        attn = F.softmax(scores, dim=-1)
        output = torch.matmul(attn, v)
        output = self._merge_heads(output)
        output = self.out_proj(output)

        present_key_value = (k, v) if use_cache else None
        return output, present_key_value

In [3]:
class TinyDecoderLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, max_seq_len: int):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.attention = CausalSelfAttentionWithKVCache(d_model=d_model, num_heads=num_heads)
        self.ln = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(
        self,
        input_ids: torch.Tensor,
        past_key_value: tuple[torch.Tensor, torch.Tensor] | None = None,
        use_cache: bool = False,
    ) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor] | None]:
        batch_size, seq_len = input_ids.shape
        past_len = 0 if past_key_value is None else past_key_value[0].size(2)

        positions = torch.arange(past_len, past_len + seq_len, device=input_ids.device)
        positions = positions.unsqueeze(0).expand(batch_size, seq_len)

        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x, present_key_value = self.attention(x, past_key_value=past_key_value, use_cache=use_cache)
        x = self.ln(x)
        logits = self.lm_head(x)
        return logits, present_key_value


@torch.no_grad()
def generate_without_cache(model: TinyDecoderLM, prompt_ids: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
    generated = prompt_ids.clone()
    for _ in range(max_new_tokens):
        logits, _ = model(generated, use_cache=False)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
    return generated


@torch.no_grad()
def generate_with_cache(model: TinyDecoderLM, prompt_ids: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
    generated = prompt_ids.clone()

    logits, cache = model(prompt_ids, use_cache=True)
    next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
    generated = torch.cat([generated, next_token], dim=1)

    for _ in range(max_new_tokens - 1):
        logits, cache = model(next_token, past_key_value=cache, use_cache=True)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)

    return generated

## 2. Decoding Algorithm With KV Cache

For a prompt of length $P$:

1. Run the prompt once through every layer.
2. Store the resulting key/value tensors from each layer.
3. Read the logits from the last prompt position and pick the next token.
4. Feed only that new token back into the model.
5. Compute its new query/key/value.
6. Append the new key/value to the cache.
7. Use the new query against the full cached keys/values.
8. Repeat until enough tokens are generated.

In practice, each transformer layer owns its own cache:
- layer 0 cache
- layer 1 cache
- layer 2 cache
- ...

So for an $L$-layer decoder, KV cache is a list of length $L$, where each element usually stores:
- keys of shape `(batch, heads, cached_seq_len, head_dim)`
- values of shape `(batch, heads, cached_seq_len, head_dim)`

The implementation below demonstrates the same idea with a single attention layer so the mechanics stay visible.

In [4]:
vocab_size = 32
model = TinyDecoderLM(vocab_size=vocab_size, d_model=24, num_heads=4, max_seq_len=64).to(device)
model.eval()

sequence = torch.tensor([[2, 5, 7, 11, 3, 19]], dtype=torch.long, device=device)
cache = None
max_diff = 0.0

with torch.no_grad():
    for step in range(1, sequence.size(1) + 1):
        full_logits, _ = model(sequence[:, :step], use_cache=False)
        full_last_logits = full_logits[:, -1, :]

        if step == 1:
            cached_logits, cache = model(sequence[:, :1], use_cache=True)
        else:
            cached_logits, cache = model(sequence[:, step - 1:step], past_key_value=cache, use_cache=True)

        cached_last_logits = cached_logits[:, -1, :]
        diff = (full_last_logits - cached_last_logits).abs().max().item()
        max_diff = max(max_diff, diff)
        print(f"step={step}, max |full - cached| = {diff:.8f}")

print(f"\nLargest difference across all steps: {max_diff:.8f}")

step=1, max |full - cached| = 0.00000000
step=2, max |full - cached| = 0.00000024
step=3, max |full - cached| = 0.00000036
step=4, max |full - cached| = 0.00000024
step=5, max |full - cached| = 0.00000026
step=6, max |full - cached| = 0.00000030

Largest difference across all steps: 0.00000036


In [5]:
prompt_ids = torch.tensor([[4, 8, 15, 16]], dtype=torch.long, device=device)
max_new_tokens = 6

without_cache = generate_without_cache(model, prompt_ids, max_new_tokens=max_new_tokens)
with_cache = generate_with_cache(model, prompt_ids, max_new_tokens=max_new_tokens)

print("Generated without cache:", without_cache.tolist())
print("Generated with cache:   ", with_cache.tolist())
print("Sequences match:", torch.equal(without_cache, with_cache))

Generated without cache: [[4, 8, 15, 16, 18, 17, 18, 17, 19, 18]]
Generated with cache:    [[4, 8, 15, 16, 18, 17, 18, 17, 19, 18]]
Sequences match: True


In [6]:
prompt_len = prompt_ids.size(1)
new_tokens = max_new_tokens

positions_without_cache = sum(prompt_len + i for i in range(new_tokens))
positions_with_cache = prompt_len + new_tokens - 1
reduction_factor = positions_without_cache / positions_with_cache

print(f"Prompt length: {prompt_len}")
print(f"New tokens: {new_tokens}")
print(f"Token positions processed without cache: {positions_without_cache}")
print(f"Token positions processed with cache:    {positions_with_cache}")
print(f"Work reduction factor: {reduction_factor:.2f}x")

Prompt length: 4
New tokens: 6
Token positions processed without cache: 39
Token positions processed with cache:    9
Work reduction factor: 4.33x


In [7]:
benchmark_model = TinyDecoderLM(vocab_size=2000, d_model=128, num_heads=8, max_seq_len=256).to(device)
benchmark_model.eval()
benchmark_prompt = torch.randint(0, 2000, (1, 48), dtype=torch.long, device=device)
benchmark_new_tokens = 48

with torch.no_grad():
    _ = generate_without_cache(benchmark_model, benchmark_prompt, max_new_tokens=4)
    _ = generate_with_cache(benchmark_model, benchmark_prompt, max_new_tokens=4)

if device.type == "cuda":
    torch.cuda.synchronize()
start = time.perf_counter()
with torch.no_grad():
    _ = generate_without_cache(benchmark_model, benchmark_prompt, max_new_tokens=benchmark_new_tokens)
if device.type == "cuda":
    torch.cuda.synchronize()
without_cache_time = time.perf_counter() - start

if device.type == "cuda":
    torch.cuda.synchronize()
start = time.perf_counter()
with torch.no_grad():
    _ = generate_with_cache(benchmark_model, benchmark_prompt, max_new_tokens=benchmark_new_tokens)
if device.type == "cuda":
    torch.cuda.synchronize()
with_cache_time = time.perf_counter() - start

print(f"Time without cache: {without_cache_time:.4f} s")
print(f"Time with cache:    {with_cache_time:.4f} s")
print(f"Measured speedup:   {without_cache_time / with_cache_time:.2f}x")

Time without cache: 0.0492 s
Time with cache:    0.0281 s
Measured speedup:   1.75x


## 3. Practical Notes

### What gets stored in real LLMs?

In a real decoder with many transformer blocks, each block stores its own past keys and values. The cache is therefore usually structured as one entry per layer.

### Why memory usage increases

KV caching trades compute for memory. As generation grows, cached sequence length grows too, so each layer keeps larger key/value tensors.

### Why prompt processing is still expensive

The prompt must still be processed once to initialize the cache. KV caching mainly saves work during the *iterative decoding* phase.

### When KV cache helps most

It helps most when:
- output sequences are long
- models are deep
- latency matters
- decoding is autoregressive

### Summary

KV cache does **not** change the math of the model. It changes **how much repeated work** we do during inference.

The output should stay the same, but inference becomes much more efficient because we reuse old keys and values instead of rebuilding them at every step.